# Q7.
```{admonition}
:class: note
Fit a neural network to the `Default` data. Use a single hidden layer with $10$ units, and dropout regularization. Compare the classification performance of your model with that of linear logistic regression.

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

In [3]:
import torch
from torchinfo import summary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
default = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/Default.csv')

In [5]:
default.head(3)

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947


In [6]:
default['student'] = (default['student'] == 'Yes').astype(float)
default['default'] = (default['default'] == 'Yes').astype(float)

In [7]:
default.head(3)

,default,student,balance,income
0,0.0,0.0,729.526495,44361.625074
1,0.0,1.0,817.180407,12106.134700
2,0.0,0.0,1073.549164,31767.138947


In [8]:
X_train, X_test, y_train, y_test = train_test_split(default.drop(columns='default'), default['default'].to_numpy(), random_state=1728)
scaler = StandardScaler()

ct = ColumnTransformer(
    transformers=[
        ('scale',StandardScaler(),['balance','income'])
    ],
    remainder='passthrough'
)

X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

In [9]:
log_reg = LogisticRegression()
log_reg.fit(X_train,y_train)
print(classification_report(y_test,log_reg.predict(X_test)))

              precision    recall  f1-score   support

         0.0       0.97      1.00      0.98      2400
         1.0       0.77      0.27      0.40       100

    accuracy                           0.97      2500
   macro avg       0.87      0.63      0.69      2500
weighted avg       0.96      0.97      0.96      2500



In [10]:
class DefaultModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(3, 10),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.sequential(x)

In [11]:
torch.manual_seed(1728)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_t = torch.tensor(X_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
default_train = TensorDataset(X_train_t,y_train_t)

train_loader = DataLoader(
    default_train,
    batch_size=64,
    shuffle=True
)

X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1).to(device)

In [12]:
default_model = DefaultModel().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(default_model.parameters(),lr=0.001)
epochs = 100

for epoch in range(epochs):
    default_model.train()
    epoch_loss=0

    for X_batch, y_batch in train_loader:
        logits = default_model(X_batch.to(device))
        loss = criterion(logits,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss+=loss.item()
        
    default_model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(default_model(X_test_t)) > 0.5
        acc = accuracy_score(y_test,preds.cpu().numpy())
        rec = recall_score(y_test,preds.cpu().numpy())
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: loss={epoch_loss/len(train_loader):.4f}, accuracy={acc:.4f}, recall={rec:.4f}')

Epoch 10: loss=0.1031, accuracy=0.9600, recall=0.0000
Epoch 20: loss=0.0853, accuracy=0.9600, recall=0.0000
Epoch 30: loss=0.0784, accuracy=0.9604, recall=0.0200
Epoch 40: loss=0.0767, accuracy=0.9660, recall=0.2100
Epoch 50: loss=0.0770, accuracy=0.9664, recall=0.2400
Epoch 60: loss=0.0761, accuracy=0.9656, recall=0.2200
Epoch 70: loss=0.0774, accuracy=0.9664, recall=0.2200
Epoch 80: loss=0.0759, accuracy=0.9668, recall=0.2500
Epoch 90: loss=0.0761, accuracy=0.9668, recall=0.2500
Epoch 100: loss=0.0762, accuracy=0.9668, recall=0.2500


In [13]:
default_model.eval()

with torch.no_grad():
    pred = torch.sigmoid(default_model(X_test_t)) > 0.5
y_pred = pred.cpu().numpy()

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         0.0       0.97      1.00      0.98      2400
         1.0       0.76      0.25      0.38       100

    accuracy                           0.97      2500
   macro avg       0.86      0.62      0.68      2500
weighted avg       0.96      0.97      0.96      2500

